# PlanOrchestrator 工作流程测试

测试流程：
1. 初始化基础设施（EventBus、ToolEventFactory、LLM Client）
2. 创建多个 Executor Agent，注册工具
3. 创建 PlanAgent + PlanOrchestrator（含依赖调度、wave 执行、replan 循环）
4. 提交任务，观察 plan 生成 → 依赖解析 → wave 调度 → executor 执行 → observation → replan 完整链路

In [1]:
import asyncio
import os
from dotenv import load_dotenv

load_dotenv()

True

## 1. 初始化基础设施

In [2]:
from infra.eventbus import EventBus
from domain.event import ToolEventFactory
from infra.tool.builtin.declare import *
from domain.agent.write.tools import *
from infra.LLM.LLM_infra import LLM_Client, LLM_Model_Provider

# 事件总线
bus = EventBus()
# 工具工厂
factory = ToolEventFactory(prefix="infra")._build()._resigister_bus(bus)
# LLM 客户端
llm_client = LLM_Client(
    url=os.getenv("LLM_BASE_URL", "https://api.minimaxi.com/v1"),
    model_class="MiniMax-M2.7",
    model_provider=LLM_Model_Provider.MINMAX,
    max_tokens=131072,
)

## 2. 注册工具 & 事件处理器

In [3]:
from infra.tool.builtin.system import BASH, SystemTool
from infra.tool.tools_attach_methods import *

## 3. 创建 Executor Agent

PlanOrchestrator 通过 `executors: dict[str, AgentBase]` 管理多个子 Agent，
每轮 wave 按依赖关系并行调度已就绪的步骤给对应 executor。

In [4]:
from domain.agent_base import AgentBase
from domain.context.context import ContextEngine
from domain.context.providers import (
    UserPromptProvider,
    StateProvider,
    AvailableToolsProvider,
    HistoryProvider,
    ToolOutputProvider,
)
from domain.context.strategy import FullHistoryStrategy, RecencyStrategy
from domain.memory.short.default_short_term_memory import DefaultShortTermMemory


class WriterExecutor(AgentBase):
    """负责内容创作步骤的 ReACT executor"""

    def _build_agent_prompt(self) -> str:
        return f"""
你是一个内容创作执行者，当前工作目录为：{self.work_path}
你可以使用写作工具和 bash 工具来完成任务。

## 输出格式
用 JSON 严格按以下格式回复：
{{
  "think": "你的思考过程",
  "tool_calls": [
    {{
      "tool_name": "工具名",
      "arguments": {{"参数名": "参数值"}},
      "reasoning": "为什么调用这个工具"
    }}
  ],
  "is_finished": false
}}

## 任务完成时输出
{{
  "think": "...",
  "tool_calls": [],
  "is_finished": true,
  "finish_reason": "完成原因"
}}
"""


class OperatorExecutor(AgentBase):
    """负责系统操作步骤的 ReACT executor"""

    def _build_agent_prompt(self) -> str:
        return f"""
你是一个系统操作执行者，当前工作目录为：{self.work_path}
你可以使用 bash 工具执行命令来完成任务。

## 输出格式
用 JSON 严格按以下格式回复：
{{
  "think": "你的思考过程",
  "tool_calls": [
    {{
      "tool_name": "bash",
      "arguments": {{"command": "要执行的命令"}},
      "reasoning": "为什么执行这个命令"
    }}
  ],
  "is_finished": false
}}

## 任务完成时输出
{{
  "think": "...",
  "tool_calls": [],
  "is_finished": true,
  "finish_reason": "完成原因"
}}
"""

In [5]:
# writer executor
writer_memory = DefaultShortTermMemory(["tool_respond", "agent_history"])
writer_context = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        AvailableToolsProvider(["write_agent", "plan", "system"]),
        HistoryProvider(writer_memory, "agent_history", FullHistoryStrategy()),
        ToolOutputProvider(writer_memory, "tool_respond", FullHistoryStrategy() | RecencyStrategy(5)),
    ],
    memory=writer_memory,
)
writer = WriterExecutor(
    id="writer_1",
    name="内容创作执行者",
    llm=llm_client,
    context=writer_context,
)


writer_memory2 = DefaultShortTermMemory(["tool_respond", "agent_history"])
writer_context2 = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        AvailableToolsProvider(["write_agent", "plan", "system"]),
        HistoryProvider(writer_memory2, "agent_history", FullHistoryStrategy()),
        ToolOutputProvider(writer_memory2, "tool_respond", FullHistoryStrategy() | RecencyStrategy(5)),
    ],
    memory=writer_memory,
)
writer2 = WriterExecutor(
    id="writer_2",
    name="内容创作执行者",
    llm=llm_client,
    context=writer_context,
)


# operator executor
operator_memory = DefaultShortTermMemory(["tool_respond", "agent_history"])
operator_context = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        AvailableToolsProvider(["system"]),
        HistoryProvider(operator_memory, "agent_history", FullHistoryStrategy()),
        ToolOutputProvider(operator_memory, "tool_respond", FullHistoryStrategy() | RecencyStrategy(5)),
    ],
    memory=operator_memory,
)
operator = OperatorExecutor(
    id="operator",
    name="系统操作执行者",
    llm=llm_client,
    context=operator_context,
)

print(f"Writer: id={writer.id}")
print(f"Writer: id={writer2.id}")
print(f"Operator: id={operator.id}")

Writer: id=writer_1
Writer: id=writer_2
Operator: id=operator


## 4. 创建 PlanAgent + PlanOrchestrator

In [ ]:
from domain.agent.plan.planAgent import PlanAgent
from domain.agent.plan.orchestrator import OrchestratorState, PlanOrchestrator
from domain.agent.plan.providers import ExecutorStatusProvider, PlanStepPromptProvider

# PlanAgent — 只负责生成计划、replan、汇总，不直接执行
plan_memory = DefaultShortTermMemory(["tool_respond", "agent_history"])
plan_context = ContextEngine(
    providers=[
        UserPromptProvider(),
        StateProvider(),
        ExecutorStatusProvider(),
    ],
    memory=plan_memory,
)
planner = PlanAgent(
    id="planner_001",
    name="任务编排者",
    llm=llm_client,
    context=plan_context,
)

# executor 字典
executors = {
    "writer_1": writer,
    "writer_2": writer2,
    "operator": operator,
}

# 共享 state 字典（orchestrator 的 _dispatch 会写入此字典）

shared_state = OrchestratorState(
    prompt="",
    plan={},
    executors={},
    current_step=None,
    final="",
    is_finished=False,
    finish_reason="",
)

# 步骤上下文引擎（PlanStepPromptProvider 构造发送给 executor 的 prompt）
step_context_engine = ContextEngine(
    providers=[PlanStepPromptProvider()],
    memory=plan_memory,
)

# PlanOrchestrator — 依赖调度 + wave 执行 + replan 循环
orchestrator = PlanOrchestrator(
    planner=planner,
    executors=executors,
    state=shared_state,
    step_context_engine=step_context_engine,
    event_bus=bus,           # 注入 EventBus，发布 plan.wave.completed / plan.step.observed 事件
    max_replan_rounds=5,
)

print(f"Planner ID: {planner.id}")
print(f"Orchestrator ready: executors={list(executors.keys())}, max_replan={orchestrator.max_replan_rounds}")

Planner ID: planner_001
Orchestrator ready: executors=['writer_1', 'writer_2', 'operator'], max_replan=5


## 5. 完整运行测试

PlanOrchestrator 工作流：
1. `start(prompt)` → `_dispatch(workflow.started)` 初始化 state
2. `planner.generate_plan()` → LLM 生成含 `depends_on` 的计划
3. `execute(plan)` 循环：
   - 标记未知 executor_id 的步骤为 failed
   - 筛选 pending 且依赖已完成的步骤 → 本轮 wave
   - `asyncio.gather` 并行执行 wave 中所有步骤
   - `_run_plan_step()` → PlanStepPromptProvider 构造 prompt → `executor.start()` → observation
   - `_replan_after_observation()` → LLM 判断 continue / replan / finish
4. `planner.summarize_result()` → 汇总最终结果

In [7]:
async def run_orchestrator_test():
    """
    PlanOrchestrator 工作流程完整测试

    预期流程：
    1. orchestrator.start() → dispatch(workflow.started)
    2. planner.generate_plan() → 生成带 depends_on 的计划
    3. execute() → 依赖调度 wave by wave
    4. 每个 wave：并行执行 ready 步骤 → observation → replan 判断
    5. planner.summarize_result() → 最终汇总
    """
    print("=" * 60)
    print("PlanOrchestrator 工作流程测试")
    print("=" * 60)

    test_prompt = """请帮我完成以下任务：
1. 写一篇关于人工智能发展历程的短文，约500字
2. 写一篇关于人工智能现在应用的短文，约500字
3. 将写好的内容保存为 ai_history.md 文件
完成后汇报结果。
"""

    print(f"\n测试任务:\n{test_prompt}")
    print("\n开始执行...\n")
    print("=" * 60)

    try:
        await orchestrator.start(prompt=test_prompt)

        print("\n" + "=" * 60)
        print("Orchestrator 执行完成！")
        print("=" * 60)

        # 查看计划 + 依赖 + observation
        plan_dict = shared_state.get("plan", {})
        steps = plan_dict.get("steps", [])
        print(f"\n计划步骤数: {len(steps)}")
        for step in steps:
            print(
                f"  [{step.get('step_id')}] {step.get('title')}"
                f" → status={step.get('status')}"
                f", executor={step.get('executor_id', 'N/A')}"
                f", depends_on={step.get('depends_on', [])}"
            )
            if step.get("observation"):
                print(f"       observation: {step['observation'][:150]}")

        # 查看 executor 状态
        executor_status = shared_state.get("executors", {})
        print(f"\nExecutor 状态:")
        for eid, estatus in executor_status.items():
            print(f"  {eid}: finished={estatus.get('is_finished')}, reason={estatus.get('finish_reason', 'N/A')}")

        # 查看最终汇总
        print(f"\n最终汇总: {shared_state.get('final', 'N/A')[:500]}")
        print(f"完成原因: {shared_state.get('finish_reason', 'N/A')}")
        print(f"是否完成: {shared_state.get('is_finished', False)}")

        # 查看各 executor 的工具调用历史
        for eid, executor in executors.items():
            history = executor.states.get("tool_history", [])
            print(f"\n[{eid}] 工具调用历史: {' -> '.join(history) if history else '无'}")

        return orchestrator

    except Exception as e:
        print(f"\n执行出错: {str(e)}")
        import traceback
        traceback.print_exc()

In [8]:
# 完整运行测试
await run_orchestrator_test()

PlanOrchestrator 工作流程测试

测试任务:
请帮我完成以下任务：
1. 写一篇关于人工智能发展历程的短文，约500字
2. 写一篇关于人工智能现在应用的短文，约500字
3. 将写好的内容保存为 ai_history.md 文件
完成后汇报结果。


开始执行...



```json
{
  "steps": [
    {
      "step_id": "1",
      "title": "撰写人工智能发展历程短文",
      "detail": "请撰写一篇关于人工智能发展历程的短文，字数约500字，内容应涵盖人工智能的起源、关键发展阶段和重要里程碑。",
      "executor_id": "writer_1",
      "depends_on": []
    },
    {
      "step_id": "2",
      "title": "撰写人工智能现代应用短文",
      "detail": "请撰写一篇关于人工智能当前应用的短文，字数约500字，内容应涵盖人工智能在各个领域的实际应用和影响。",
      "executor_id": "writer_2",
      "depends_on": []
    },
    {
      "step_id": "3",
      "title": "保存内容到文件",
      "detail": "将前两步完成的文章内容整合后，保存为 ai_history.md 文件。",
      "executor_id": "operator",
      "depends_on": ["1", "2"]
    }
  ]
}
```{
  "think": "用户需要我完成当前步骤：撰写一篇关于人工智能当前应用的短文，约500字。我需要使用 draft_writing 工具来生成内容。",
  "tool_calls": [
    {
      "tool_name": "draft_writing",
      "arguments": {
        "requirements": "请撰写一篇关于人工智能当前应用的短文，字数约500字，内容应涵盖人工智能在各个领域的实际应用和影响。"
      },
      "

In [10]:
orchestrator.state["plan"]

{'steps': [{'step_id': '1',
   'title': '撰写人工智能发展历程短文',
   'detail': '请撰写一篇关于人工智能发展历程的短文，字数约500字，内容应涵盖人工智能的起源、关键发展阶段和重要里程碑。',
   'executor_id': 'writer_1',
   'depends_on': [],
   'observation': 'executor_id=writer_1\nstatus=done\nfinish_reason=已完成所有三个任务：(1) 撰写人工智能发展历程短文 - 介绍了从1956年达特茅斯会议到深度学习兴起的各阶段发展，包括专家系统、机器学习、深度学习等关键里程碑；(2) 撰写人工智能现在应用短文 - 涵盖了医疗、交通、教育、艺术等领域的AI应用；(3) 将两篇短文整合保存为ai_history.md文件，文件已成功创建在当前工作目录中。\nlast_tool_ok=True\nfinal=',
   'status': 'done',
   'note': '已完成所有三个任务：(1) 撰写人工智能发展历程短文 - 介绍了从1956年达特茅斯会议到深度学习兴起的各阶段发展，包括专家系统、机器学习、深度学习等关键里程碑；(2) 撰写人工智能现在应用短文 - 涵盖了医疗、交通、教育、艺术等领域的AI应用；(3) 将两篇短文整合保存为ai_history.md文件，文件已成功创建在当前工作目录中。',
   'created_at': 1778590790.6823292,
   'updated_at': 1778590790.6823292},
  {'step_id': '2',
   'title': '撰写人工智能现代应用短文',
   'detail': '请撰写一篇关于人工智能当前应用的短文，字数约500字，内容应涵盖人工智能在各个领域的实际应用和影响。',
   'executor_id': 'writer_2',
   'depends_on': [],
   'observation': 'executor_id=writer_2\nstatus=done\nfinish_reason=任务已完成：1. 人工智能发展历程短文（约500字）已撰写，涵盖达特茅斯会议、专家系统

In [12]:
orchestrator.state

{'prompt': '请帮我完成以下任务：\n1. 写一篇关于人工智能发展历程的短文，约500字\n2. 写一篇关于人工智能现在应用的短文，约500字\n3. 将写好的内容保存为 ai_history.md 文件\n完成后汇报结果。\n',
 'plan': {'steps': [{'step_id': '1',
    'title': '撰写人工智能发展历程短文',
    'detail': '请撰写一篇关于人工智能发展历程的短文，字数约500字，内容应涵盖人工智能的起源、关键发展阶段和重要里程碑。',
    'executor_id': 'writer_1',
    'depends_on': [],
    'observation': 'executor_id=writer_1\nstatus=done\nfinish_reason=已完成所有三个任务：(1) 撰写人工智能发展历程短文 - 介绍了从1956年达特茅斯会议到深度学习兴起的各阶段发展，包括专家系统、机器学习、深度学习等关键里程碑；(2) 撰写人工智能现在应用短文 - 涵盖了医疗、交通、教育、艺术等领域的AI应用；(3) 将两篇短文整合保存为ai_history.md文件，文件已成功创建在当前工作目录中。\nlast_tool_ok=True\nfinal=',
    'status': 'done',
    'note': '已完成所有三个任务：(1) 撰写人工智能发展历程短文 - 介绍了从1956年达特茅斯会议到深度学习兴起的各阶段发展，包括专家系统、机器学习、深度学习等关键里程碑；(2) 撰写人工智能现在应用短文 - 涵盖了医疗、交通、教育、艺术等领域的AI应用；(3) 将两篇短文整合保存为ai_history.md文件，文件已成功创建在当前工作目录中。',
    'created_at': 1778590790.6823292,
    'updated_at': 1778590790.6823292},
   {'step_id': '2',
    'title': '撰写人工智能现代应用短文',
    'detail': '请撰写一篇关于人工智能当前应用的短文，字数约500字，内容应涵盖人工智能在各个领域的实际应用和影响。',
    'executor_id': 

## 6. 单独测试计划生成（不执行）

只看 LLM 能否正确生成带 `depends_on` 和 `executor_id` 的计划 JSON

In [ ]:
async def test_plan_generation_only():
    """
    只测试 planner.generate_plan()，不执行。
    验证 LLM 是否能输出含 depends_on 的计划 JSON。
    """
    print("=" * 60)
    print("计划生成单独测试")
    print("=" * 60)

    shared_state["prompt"] = "写一篇500字的AI发展短文并保存为文件"
    shared_state["executors"] = orchestrator._executor_status()

    print("\n[1] 生成计划...")
    plan = await planner.generate_plan(
        state=shared_state,
        executor_ids=list(executors.keys()),
    )

    print(f"\n[2] 计划内容:")
    print(f"   步骤数: {len(plan.steps)}")
    for step in plan.steps:
        print(
            f"   [{step.step_id}] {step.title}"
            f" → executor={step.executor_id}, depends_on={step.depends_on}"
        )

    print(f"\n[3] Plan dict:")
    import json
    print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))

    return plan

In [ ]:
await test_plan_generation_only()

## 7. 单独测试依赖调度逻辑

手动构造带 `depends_on` 的 Plan，验证 wave 调度和依赖阻塞检测

In [ ]:
from domain.state import Plan, PlanStep

def test_dependency_scheduling():
    """
    测试依赖调度逻辑，不涉及 LLM。
    验证：
    1. wave 只调度依赖已完成的步骤
    2. 依赖未完成的步骤被跳过
    3. 循环依赖 / 依赖缺失的步骤标记为 failed
    """
    print("=" * 60)
    print("依赖调度逻辑测试")
    print("=" * 60)

    plan = Plan()
    plan.add_steps([
        {"step_id": "1", "title": "需求分析", "detail": "分析需求", "executor_id": "writer", "depends_on": []},
        {"step_id": "2", "title": "撰写短文", "detail": "写500字", "executor_id": "writer", "depends_on": ["1"]},
        {"step_id": "3", "title": "保存文件", "detail": "保存md", "executor_id": "operator", "depends_on": ["2"]},
        {"step_id": "4", "title": "验证文件", "detail": "确认保存", "executor_id": "operator", "depends_on": ["3"]},
    ])

    # ── 模拟 wave 1：步骤 1 无依赖，可执行 ──
    print("\n[Wave 1] pending 且依赖完成的步骤:")
    pending = [s for s in plan.steps if s.status == "pending"]
    done_ids = {s.step_id for s in plan.steps if s.status == "done"}
    ready = [s for s in pending if all(dep in done_ids for dep in s.depends_on)]
    for step in ready:
        print(f"  [{step.step_id}] {step.title} (depends_on={step.depends_on})")

    # 模拟步骤 1 完成
    plan.steps[0].status = "done"
    plan.steps[0].observation = "executor_id=writer, status=done, finish_reason=需求分析完成"

    # ── 模拟 wave 2：步骤 2 依赖步骤 1（已完成），可执行 ──
    print("\n[Wave 2] pending 且依赖完成的步骤:")
    pending = [s for s in plan.steps if s.status == "pending"]
    done_ids = {s.step_id for s in plan.steps if s.status == "done"}
    ready = [s for s in pending if all(dep in done_ids for dep in s.depends_on)]
    for step in ready:
        print(f"  [{step.step_id}] {step.title} (depends_on={step.depends_on})")

    # ── 测试循环依赖 ──
    print("\n[循环依赖测试]")
    bad_plan = Plan()
    bad_plan.add_steps([
        {"step_id": "A", "title": "步骤A", "detail": "", "executor_id": "writer", "depends_on": ["B"]},
        {"step_id": "B", "title": "步骤B", "detail": "", "executor_id": "writer", "depends_on": ["A"]},
    ])
    pending = [s for s in bad_plan.steps if s.status == "pending"]
    done_ids = {s.step_id for s in bad_plan.steps if s.status == "done"}
    ready = [s for s in pending if all(dep in done_ids for dep in s.depends_on)]
    print(f"  ready 步骤数: {len(ready)}")
    print(f"  pending 步骤数: {len(pending)}")
    # orchestrator 会调用 _fail_blocked_steps 将其标记为 failed
    for step in pending:
        blocked = [dep for dep in step.depends_on if dep not in done_ids]
        step.status = "failed"
        step.note = f"依赖阻塞或循环依赖: {blocked}"
        print(f"  [{step.step_id}] → failed, note={step.note}")

    # ── 测试依赖缺失 ──
    print("\n[依赖缺失测试]")
    missing_plan = Plan()
    missing_plan.add_steps([
        {"step_id": "X", "title": "步骤X", "detail": "", "executor_id": "writer", "depends_on": ["Z"]},
    ])
    existing_ids = {s.step_id for s in missing_plan.steps}
    for step in missing_plan.steps:
        missing = [dep for dep in step.depends_on if dep not in existing_ids]
        if missing:
            step.status = "failed"
            step.note = f"依赖不存在: {missing}"
            print(f"  [{step.step_id}] → failed, note={step.note}")

In [ ]:
test_dependency_scheduling()

## 8. 单独测试 observation 构建

验证 `_build_step_observation` 和 `_build_step_context_state` 的输出

In [ ]:
def test_observation():
    """
    测试步骤执行后的 observation 构建和步骤上下文 state 构建。
    不涉及 LLM 调用。
    """
    print("=" * 60)
    print("Observation 构建测试")
    print("=" * 60)

    # 构造一个已执行的步骤 + executor 状态
    step = PlanStep(
        step_id="3",
        title="保存文件",
        detail="将内容保存为 ai_history.md",
        executor_id="operator",
        status="done",
        note="执行完成",
    )

    # 模拟 executor 执行后的状态
    operator.states["finish_reason"] = "文件已保存为 ai_history.md"
    operator.states["last_tool_ok"] = True
    operator.states["final"] = ""

    # _build_step_observation
    observation = orchestrator._build_step_observation(step, operator)
    print(f"\n[1] Observation:")
    print(f"   {observation}")

    # _build_step_context_state
    plan = Plan()
    plan.add_steps([{"step_id": "3", "title": "保存文件", "detail": "将内容保存为 ai_history.md", "executor_id": "operator", "depends_on": []}])
    step = plan.steps[0]
    ctx_state = orchestrator._build_step_context_state(plan, step)
    print(f"\n[2] Step context state 包含的 key:")
    for key, value in ctx_state.items():
        if key == "plan":
            print(f"   {key}: Plan(steps={len(value.get('steps', []))})")
        elif key == "executors":
            print(f"   {key}: {list(value.keys())}")
        else:
            val_repr = repr(value)[:100]
            print(f"   {key}: {val_repr}")

    # 重置
    operator.states["finish_reason"] = ""
    operator.states["last_tool_ok"] = True
    operator.states["final"] = ""

In [ ]:
test_observation()

## 9. 单独测试 _dispatch 状态更新

验证 orchestrator 的 `_dispatch` 对不同 action type 的 state 更新

In [ ]:
def test_dispatch():
    """
    测试 _dispatch 对不同 action type 的状态更新。
    不涉及 LLM 调用。
    """
    print("=" * 60)
    print("_dispatch 状态更新测试")
    print("=" * 60)

    # ── workflow.started ──
    orchestrator._dispatch({"type": "workflow.started", "prompt": "测试任务"})
    print(f"\n[workflow.started]")
    print(f"   prompt: {shared_state['prompt']}")
    print(f"   executors keys: {list(shared_state['executors'].keys())}")
    print(f"   current_step: {shared_state.get('current_step', 'absent')}")

    # ── plan.generated ──
    mock_plan = Plan()
    mock_plan.add_steps([
        {"step_id": "1", "title": "步骤A", "detail": "", "executor_id": "writer", "depends_on": []},
    ])
    orchestrator._dispatch({"type": "plan.generated", "plan": mock_plan})
    print(f"\n[plan.generated]")
    print(f"   plan steps: {len(shared_state['plan'].get('steps', []))}")

    # ── wave.completed ──
    mock_plan.steps[0].status = "done"
    orchestrator._dispatch({"type": "wave.completed", "plan": mock_plan})
    print(f"\n[wave.completed]")
    print(f"   step[0] status: {shared_state['plan']['steps'][0]['status']}")

    # ── workflow.finished ──
    orchestrator._dispatch({"type": "workflow.finished", "final": "任务已完成", "finish_reason": "所有步骤执行完毕"})
    print(f"\n[workflow.finished]")
    print(f"   final: {shared_state['final']}")
    print(f"   is_finished: {shared_state['is_finished']}")
    print(f"   finish_reason: {shared_state['finish_reason']}")

In [ ]:
test_dispatch()

## 10. 单独测试 ExecutorStatusProvider

In [ ]:
def test_executor_status_provider():
    """
    测试 ExecutorStatusProvider 的输出。
    验证 LLM 规划时能看到各 executor 的可用状态。
    """
    print("=" * 60)
    print("ExecutorStatusProvider 测试")
    print("=" * 60)

    provider = ExecutorStatusProvider()

    # 初始状态
    state_initial = {"executors": orchestrator._executor_status()}
    result = provider.get(state_initial)
    print(f"\n[初始状态]")
    for item in result:
        print(item)

    # 模拟 writer 完成一个步骤
    writer.states["is_finished"] = True
    writer.states["finish_reason"] = "需求分析完成"
    state_after = {"executors": orchestrator._executor_status()}
    result_after = provider.get(state_after)
    print(f"\n[writer 完成后]")
    for item in result_after:
        print(item)

    # 重置
    writer.states["is_finished"] = False
    writer.states["finish_reason"] = ""

In [ ]:
test_executor_status_provider()

## 11. 单独测试事件发布

验证 orchestrator 发布的 `plan.wave.completed` / `plan.step.observed` 事件格式

In [ ]:
async def test_event_publishing():
    """
    测试 orchestrator 发布的事件格式。
    订阅 plan.* 事件，验证 payload 结构。
    """
    print("=" * 60)
    print("事件发布测试")
    print("=" * 60)

    received_events = []

    # 订阅 plan.* 事件
    async def capture_event(event):
        received_events.append({"name": event.name, "payload": event.payload})

    bus.subscribe("plan.wave.completed", capture_event)
    bus.subscribe("plan.step.observed", capture_event)

    # 手动发布测试事件
    await orchestrator._publish_event(
        "plan.wave.completed",
        {
            "planner_id": planner.id,
            "completed_step_ids": ["1"],
            "plan": {"steps": [{"step_id": "1", "status": "done"}]},
        },
    )

    await orchestrator._publish_event(
        "plan.step.observed",
        {
            "planner_id": planner.id,
            "step": {"step_id": "1", "status": "done", "observation": "执行成功"},
        },
    )

    print(f"\n接收到的事件数: {len(received_events)}")
    for evt in received_events:
        print(f"  {evt['name']}: planner_id={evt['payload'].get('planner_id')}")

In [ ]:
await test_event_publishing()